# Scalable Inception Graph Neural Networks (SIGN) on Cora

Node Classification on Cora (Planetoid): Precomputing multi-scale graph diffusion operators for fast parallel GNN training. This notebook implements the approach with `SIGN` inside a `K3SIGN` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `SIGN` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node.datasets import Planetoid

title = "Scalable Inception Graph Neural Networks (SIGN) on Cora"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora")
data = dataset[0]

# 2. SIGN Architecture (Dense projections of precomputed multi-hop features)
class K3SIGN(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels, K=2):
        super().__init__()
        self.lins = [layers.Dense(hidden_channels) for _ in range(K + 1)]
        self.lin = layers.Dense(out_channels)

    def call(self, xs):
        hs = [ops.relu(lin(x)) for lin, x in zip(self.lins, xs)]
        h = ops.concatenate(hs, axis=-1)
        return self.lin(h)

k3_model = K3SIGN(dataset.num_features, 64, dataset.num_classes, K=2)

# Multi-hop precomputed feature list (x0, x1, x2)
xs = [data.x, data.x, data.x]
out = k3_model(xs)
print(f"SIGN forward pass prediction shape: {out.shape}")

print("\n✓ K3-Node SIGN execution completed successfully!")